# Weather Pipeline: Raw API to Reduced Dataset

Build one notebook flow that starts with Open-Meteo weather data and ends with a reduced feature dataset for household load prediction.

## Notebook Roadmap

This notebook is organized into seven simple steps:

1. Load paths, location settings, and shared weather variables.
2. Load the household timestamp calendar and the first target load.
3. Fetch raw historical weather data.
4. Convert raw API output into one clean 15-minute UTC weather table.
5. Create model-ready weather features.
6. Reduce the feature set for `residential1`.
7. Reuse the same reduced structure for future API calls and save outputs.

The notebook is meant as a clear workflow first. Later, the same logic can move into reusable Python modules.

In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [3]:
PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "config.json"
HOUSEHOLD_DATA_PATH = PROJECT_ROOT / "data" / "household_data_15min_singleindex.csv"
RAW_WEATHER_OUTPUT_PATH = PROJECT_ROOT / "data" / "weather_raw_hourly.csv"
FULL_WEATHER_OUTPUT_PATH = PROJECT_ROOT / "data" / "weather_full_15min.csv"
REDUCED_WEATHER_OUTPUT_PATH = PROJECT_ROOT / "data" / "weather_reduced_residential1.csv"
REDUCTION_SPEC_PATH = PROJECT_ROOT / "models" / "weather_feature_spec_residential1.pkl"

with CONFIG_PATH.open("r", encoding="utf-8") as file_handle:
    config = json.load(file_handle)

LATITUDE = config["lat"]
LONGITUDE = config["lon"]

LATITUDE, LONGITUDE

(47.659216, 9.1750718)

## Step 1: Define Shared Weather Variables

Keep one shared weather schema that can be used for historical data and future API calls.

In [4]:
COMMON_HOURLY_WEATHER_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "snow_depth",
    "weather_code",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high",
    "shortwave_radiation",
    "direct_radiation",
    "diffuse_radiation",
    "global_tilted_irradiance",
    "sunshine_duration",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "et0_fao_evapotranspiration",
    "vapour_pressure_deficit",
]

DERIVED_WEATHER_FEATURES = [
    "heating_degree_18c",
    "cooling_degree_22c",
    "is_raining",
    "is_snowing",
    "wind_u_10m",
    "wind_v_10m",
    "is_dark",
]

COMMON_HOURLY_WEATHER_VARS

['temperature_2m',
 'relative_humidity_2m',
 'dew_point_2m',
 'apparent_temperature',
 'precipitation',
 'rain',
 'snowfall',
 'snow_depth',
 'weather_code',
 'pressure_msl',
 'surface_pressure',
 'cloud_cover',
 'cloud_cover_low',
 'cloud_cover_mid',
 'cloud_cover_high',
 'shortwave_radiation',
 'direct_radiation',
 'diffuse_radiation',
 'global_tilted_irradiance',
 'sunshine_duration',
 'wind_speed_10m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'et0_fao_evapotranspiration',
 'vapour_pressure_deficit']

## Step 2: Load Household Calendar And Target

Use the household timestamps as the calendar we want to match. Start with `residential1` as the first target for feature reduction.

In [5]:
household_df = pd.read_csv(HOUSEHOLD_DATA_PATH, parse_dates=["utc_timestamp"])
household_df["utc_timestamp"] = pd.to_datetime(household_df["utc_timestamp"], utc=True)
household_df = household_df.sort_values("utc_timestamp").reset_index(drop=True)

household_calendar = household_df[["utc_timestamp"]].drop_duplicates().reset_index(drop=True)

raw_target_cols = [
    "DE_KN_residential1_grid_import",
    "DE_KN_residential1_grid_export",
    "DE_KN_residential1_pv",
]
available_target_cols = [column for column in raw_target_cols if column in household_df.columns]
household_df[available_target_cols] = household_df[available_target_cols].diff()

household_df["residential1_load"] = 0.0
if "DE_KN_residential1_grid_import" in household_df.columns:
    household_df["residential1_load"] += household_df["DE_KN_residential1_grid_import"].fillna(0.0)
if "DE_KN_residential1_grid_export" in household_df.columns:
    household_df["residential1_load"] -= household_df["DE_KN_residential1_grid_export"].fillna(0.0)
if "DE_KN_residential1_pv" in household_df.columns:
    household_df["residential1_load"] += household_df["DE_KN_residential1_pv"].fillna(0.0)

household_calendar.head()

,utc_timestamp
0,2014-12-11 17:45:00+00:00
1,2014-12-11 18:00:00+00:00
2,2014-12-11 18:15:00+00:00
3,2014-12-11 18:30:00+00:00
4,2014-12-11 18:45:00+00:00


## Step 3: Fetch Raw Historical Weather Data

Fetch raw hourly data first. Later this should call one shared weather function for both historical backfill and future forecast runs.

In [6]:
import requests

ARCHIVE_ENDPOINT = "https://archive-api.open-meteo.com/v1/archive"
REQUEST_TIMEOUT_SECONDS = 60

historical_start_date = household_calendar["utc_timestamp"].min().date()
historical_end_date = household_calendar["utc_timestamp"].max().date()

month_periods = pd.period_range(start=historical_start_date, end=historical_end_date, freq="M")
chunk_frames = []

for period in month_periods:
    chunk_start = max(historical_start_date, period.start_time.date())
    chunk_end = min(historical_end_date, period.end_time.date())

    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": chunk_start.isoformat(),
        "end_date": chunk_end.isoformat(),
        "hourly": ",".join(COMMON_HOURLY_WEATHER_VARS),
        "timezone": "UTC",
    }

    if "global_tilted_irradiance" in COMMON_HOURLY_WEATHER_VARS and {"tilt", "azimuth"}.issubset(config):
        params["tilt"] = config["tilt"]
        params["azimuth"] = config["azimuth"]

    response = requests.get(ARCHIVE_ENDPOINT, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    payload = response.json()

    if isinstance(payload, dict) and payload.get("error"):
        reason = payload.get("reason", "Unknown API error")
        raise ValueError(f"Open-Meteo archive API error for {chunk_start} -> {chunk_end}: {reason}")

    hourly_payload = payload.get("hourly", {})
    if "time" not in hourly_payload:
        raise ValueError(
            f"Missing 'time' in hourly payload for {chunk_start} -> {chunk_end}. "
            f"Available keys: {list(hourly_payload.keys())}"
        )

    chunk_df = pd.DataFrame(hourly_payload)
    chunk_df["time"] = pd.to_datetime(chunk_df["time"], utc=True, errors="coerce")
    chunk_df = chunk_df.dropna(subset=["time"]).sort_values("time")

    for weather_col in COMMON_HOURLY_WEATHER_VARS:
        if weather_col not in chunk_df.columns:
            chunk_df[weather_col] = np.nan

    chunk_df = chunk_df[["time", *COMMON_HOURLY_WEATHER_VARS]].copy()
    chunk_df["source"] = "open_meteo_archive"

    chunk_frames.append(chunk_df)
    print(f"Fetched {chunk_start} -> {chunk_end}: {len(chunk_df)} rows")

if not chunk_frames:
    raise ValueError("No historical weather chunks were fetched.")

raw_weather_df = pd.concat(chunk_frames, ignore_index=True)
raw_weather_df = raw_weather_df.sort_values("time").drop_duplicates(subset=["time"], keep="first")
raw_weather_df = raw_weather_df.reset_index(drop=True)

RAW_WEATHER_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
raw_weather_df.to_csv(RAW_WEATHER_OUTPUT_PATH, index=False)

print(
    f"Historical range: {historical_start_date} -> {historical_end_date} | "
    f"rows={len(raw_weather_df)} | cols={raw_weather_df.shape[1]}"
)
print(f"Saved raw hourly weather data to: {RAW_WEATHER_OUTPUT_PATH}")
raw_weather_df.head()

Fetched 2014-12-11 -> 2014-12-31: 504 rows
Fetched 2015-01-01 -> 2015-01-31: 744 rows
Fetched 2015-02-01 -> 2015-02-28: 672 rows
Fetched 2015-03-01 -> 2015-03-31: 744 rows
Fetched 2015-04-01 -> 2015-04-30: 720 rows
Fetched 2015-05-01 -> 2015-05-31: 744 rows
Fetched 2015-06-01 -> 2015-06-30: 720 rows
Fetched 2015-07-01 -> 2015-07-31: 744 rows
Fetched 2015-08-01 -> 2015-08-31: 744 rows
Fetched 2015-09-01 -> 2015-09-30: 720 rows
Fetched 2015-10-01 -> 2015-10-31: 744 rows
Fetched 2015-11-01 -> 2015-11-30: 720 rows
Fetched 2015-12-01 -> 2015-12-31: 744 rows
Fetched 2016-01-01 -> 2016-01-31: 744 rows
Fetched 2016-02-01 -> 2016-02-29: 696 rows
Fetched 2016-03-01 -> 2016-03-31: 744 rows
Fetched 2016-04-01 -> 2016-04-30: 720 rows
Fetched 2016-05-01 -> 2016-05-31: 744 rows
Fetched 2016-06-01 -> 2016-06-30: 720 rows
Fetched 2016-07-01 -> 2016-07-31: 744 rows
Fetched 2016-08-01 -> 2016-08-31: 744 rows
Fetched 2016-09-01 -> 2016-09-30: 720 rows
Fetched 2016-10-01 -> 2016-10-31: 744 rows
Fetched 201

,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,weather_code,...,direct_radiation,diffuse_radiation,global_tilted_irradiance,sunshine_duration,wind_speed_10m,wind_direction_10m,wind_gusts_10m,et0_fao_evapotranspiration,vapour_pressure_deficit,source
0,2014-12-11 00:00:00+00:00,4.2,88,2.4,1.1,0.1,0.1,0.0,0.0,51,...,0.0,0.0,0.0,0.0,9.7,211,18.4,0.0,0.10,open_meteo_archive
1,2014-12-11 01:00:00+00:00,4.5,88,2.8,1.4,0.2,0.2,0.0,0.0,51,...,0.0,0.0,0.0,0.0,10.5,232,22.0,0.0,0.10,open_meteo_archive
2,2014-12-11 02:00:00+00:00,4.6,90,3.1,1.1,0.3,0.3,0.0,0.0,51,...,0.0,0.0,0.0,0.0,12.9,240,26.3,0.0,0.09,open_meteo_archive
3,2014-12-11 03:00:00+00:00,4.6,91,3.3,1.0,0.3,0.3,0.0,0.0,51,...,0.0,0.0,0.0,0.0,14.2,240,29.5,0.0,0.07,open_meteo_archive
4,2014-12-11 04:00:00+00:00,4.4,93,3.3,0.8,0.3,0.3,0.0,0.0,51,...,0.0,0.0,0.0,0.0,13.9,239,30.2,0.0,0.06,open_meteo_archive


## Step 4: Build The Full 15-Minute Weather Dataset

Turn raw API output into one clean dataset with UTC timestamps every 15 minutes. This is the full weather table before feature reduction.

In [ ]:
# Start from raw hourly weather data and standardize timestamps.
working_df = raw_weather_df.copy()
working_df["time"] = pd.to_datetime(working_df["time"], utc=True, errors="coerce")
working_df = working_df.dropna(subset=["time"]).sort_values("time")
working_df = working_df.drop_duplicates(subset=["time"], keep="first")
working_df = working_df.set_index("time")

weather_cols = [col for col in COMMON_HOURLY_WEATHER_VARS if col in working_df.columns]
for weather_col in weather_cols:
    working_df[weather_col] = pd.to_numeric(working_df[weather_col], errors="coerce")

calendar_index = pd.DatetimeIndex(household_calendar["utc_timestamp"]).tz_convert("UTC")
calendar_index = calendar_index.sort_values().unique()
calendar_index = pd.DatetimeIndex(calendar_index)

combined_index = working_df.index.union(calendar_index)
full_weather_indexed = working_df[weather_cols].reindex(combined_index).sort_index()
full_weather_indexed = full_weather_indexed.interpolate(method="time", limit_direction="both")
full_weather_indexed = full_weather_indexed.ffill().bfill()

full_weather_df = full_weather_indexed.reindex(calendar_index).reset_index()
full_weather_df = full_weather_df.rename(columns={"index": "utc_timestamp"})
full_weather_df["source"] = "open_meteo_archive_interpolated"

print(f"Built full_weather_df with {len(full_weather_df)} rows and {full_weather_df.shape[1]} columns")
print(f"Expected rows from household calendar: {len(calendar_index)}")
print(f"Duplicate timestamps in full_weather_df: {full_weather_df['utc_timestamp'].duplicated().sum()}")

FULL_WEATHER_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
full_weather_df.to_csv(FULL_WEATHER_OUTPUT_PATH, index=False)
print(f"Saved full 15-minute weather data to: {FULL_WEATHER_OUTPUT_PATH}")

full_weather_df.head()

Built full_weather_df with 153810 rows and 27 columns
Expected rows from household calendar: 153810
Duplicate timestamps in full_weather_df: 0
Saved full 15-minute weather data to: /Users/joscham/dsai/repos/hems-automation/data/weather_full_15min.csv


,utc_timestamp,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,weather_code,...,direct_radiation,diffuse_radiation,global_tilted_irradiance,sunshine_duration,wind_speed_10m,wind_direction_10m,wind_gusts_10m,et0_fao_evapotranspiration,vapour_pressure_deficit,source
0,2014-12-11 17:45:00+00:00,4.875,79.5,1.650,-0.625,0.30,0.30,0.0,0.0,51.0,...,0.0,0.0,0.0,0.0,25.125,237.50,55.925,0.0225,0.1775,open_meteo_archive_interpolated
1,2014-12-11 18:00:00+00:00,4.800,80.0,1.700,-0.700,0.30,0.30,0.0,0.0,51.0,...,0.0,0.0,0.0,0.0,25.100,238.00,58.000,0.0200,0.1700,open_meteo_archive_interpolated
2,2014-12-11 18:15:00+00:00,4.825,80.0,1.725,-0.600,0.35,0.35,0.0,0.0,51.5,...,0.0,0.0,0.0,0.0,24.675,238.25,58.450,0.0200,0.1700,open_meteo_archive_interpolated
3,2014-12-11 18:30:00+00:00,4.850,80.0,1.750,-0.500,0.40,0.40,0.0,0.0,52.0,...,0.0,0.0,0.0,0.0,24.250,238.50,58.900,0.0200,0.1700,open_meteo_archive_interpolated
4,2014-12-11 18:45:00+00:00,4.875,80.0,1.775,-0.400,0.45,0.45,0.0,0.0,52.5,...,0.0,0.0,0.0,0.0,23.825,238.75,59.350,0.0200,0.1700,open_meteo_archive_interpolated


## Step 5: Create Model-Ready Weather Features

Add a small set of useful features that can also be created later from future forecast data.

In [ ]:
print("Derived features planned for the first version:")
for feature_name in DERIVED_WEATHER_FEATURES:
    print(f"- {feature_name}")

print("TODO: create weather_features_df from full_weather_df")

weather_features_df = None

Derived features planned for the first version:
- heating_degree_18c
- cooling_degree_22c
- is_raining
- is_snowing
- wind_u_10m
- wind_v_10m
- is_dark
TODO: create weather_features_df from full_weather_df


## Step 6: Reduce The Feature Set For Residential1

Use historical data to decide which weather features matter most for `residential1_load`. Save the selected columns so the same reduced dataset can be created later.

In [ ]:
REDUCTION_TARGET = "residential1_load"

print(f"Current reduction target: {REDUCTION_TARGET}")
print("TODO: merge weather_features_df with the target series")
print("TODO: fit a time-aware feature selector")
print("TODO: save selected feature names and order")

selected_feature_columns = []

Current reduction target: residential1_load
TODO: merge weather_features_df with the target series
TODO: fit a time-aware feature selector
TODO: save selected feature names and order


## Step 7: Reuse The Same Reduced Structure For Future API Calls

A future forecast call should go through the same cleaning and feature steps, then keep only the saved selected columns.

In [ ]:
print("Future-call workflow:")
print("1. fetch forecast weather data")
print("2. build the same 15-minute UTC weather table")
print("3. create the same derived weather features")
print("4. keep only selected_feature_columns")

future_reduced_df = None

Future-call workflow:
1. fetch forecast weather data
2. build the same 15-minute UTC weather table
3. create the same derived weather features
4. keep only selected_feature_columns


## Save Outputs And Final Checks

Save three outputs:

- the full weather dataset
- the reduced weather dataset
- the saved reduction rule

Before saving, check that timestamps are in UTC, there are no duplicates, and the reduced historical and future datasets have the same columns.

In [ ]:
OUTPUT_PATHS = {
    "raw_weather_dataset": RAW_WEATHER_OUTPUT_PATH,
    "full_weather_dataset": FULL_WEATHER_OUTPUT_PATH,
    "reduced_weather_dataset": REDUCED_WEATHER_OUTPUT_PATH,
    "reduction_spec": REDUCTION_SPEC_PATH,
}

OUTPUT_PATHS

{'full_weather_dataset': PosixPath('/Users/joscham/dsai/repos/hems-automation/data/weather_full_15min.csv'),
 'reduced_weather_dataset': PosixPath('/Users/joscham/dsai/repos/hems-automation/data/weather_reduced_residential1.csv'),
 'reduction_spec': PosixPath('/Users/joscham/dsai/repos/hems-automation/models/weather_feature_spec_residential1.pkl')}